# **Bài thực hành 2. Làm sạch dữ liệu (Quản lý dữ liệu)**

## **2.4. Bài 1. Xác minh tính toàn vẹn của dữ liệu**

## Import thư viện cần thiết.

In [ ]:
import pandas as pd
import numpy as np

### Đọc file csv từ tập dữ liệu Credit Card dataset. Kiểm tra số dòng và số cột

In [ ]:
df_b2 = pd.read_csv('/content/drive/MyDrive/credit_card.csv')

In [ ]:
shape = df_b2.shape
print(f"Dữ liệu có {shape[0]} dòng và {shape[1]} cột.")

Dữ liệu có 30000 dòng và 25 cột.


### Kiểm tra tên các cột bằng cách chạy lệnh .columns. Giải thích ý nghĩa từng thuộc tính trong Markdown tiếp theo.

In [ ]:
print(df_b2.columns)

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='object')


Tập dữ liệu này cung cấp một cái nhìn toàn diện về hồ sơ và lịch sử tài chính của từng khách hàng. Đầu tiên, cột `ID` đóng vai trò là mã định danh duy nhất cho mỗi tài khoản. Biến `LIMIT_BAL` cho biết tổng hạn mức tín dụng được cấp, bao gồm cả tín dụng cá nhân và gia đình của khách hàng đó. Các thông tin nhân khẩu học cơ bản được ghi nhận qua các biến `SEX` (Giới tính) , `EDUCATION` (Trình độ học vấn) , `MARRIAGE` (Tình trạng hôn nhân) và `AGE` (Độ tuổi). Hành vi trả nợ của khách hàng trong giai đoạn từ tháng 4 đến tháng 9 năm 2005 được theo dõi chi tiết qua các cột từ `PAY_1` đến `PAY_6`, sử dụng thang đo từ -1 (trả đúng hạn) đến các số nguyên dương (ví dụ: 1 là chậm một tháng, 8 là chậm tám tháng). Bên cạnh đó, số tiền ghi trên hóa đơn hàng tháng được lưu trữ trong các biến từ `BILL_AMT1` đến `BILL_AMT6`, trong khi số tiền thực tế khách hàng đã thanh toán trong các tháng trước đó được ghi lại từ `PAY_AMT1` đến `PAY_AMT6`. Cuối cùng, cột `default payment next month` đóng vai trò là biến mục tiêu nhị phân, cho biết liệu chủ tài khoản có bị vỡ nợ (không thực hiện được khoản thanh toán tối thiểu) vào tháng tiếp theo hay không.

### Kiểm tra 5 dòng đầu tiên trong tập dữ liệu.

In [ ]:
df_b2.head(5)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,798fc410-45c1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,8a8c8f3b-8eb4,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,85698822-43f5,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,0737c11b-be42,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,3b7f77cc-dbc0,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### Kiểm tra tính duy nhất của cột ID

In [ ]:
unique_ids = df_b2['ID'].nunique()
print(f"Số lượng ID duy nhất là: {unique_ids}")

Số lượng ID duy nhất là: 29687


Do tổng số dòng là 30.000 nhưng chỉ có 29.687 ID duy nhất, điều này cho thấy có 313 ID bị lặp lại. Dữ liệu cần được xử lý để loại bỏ các bản ghi trùng lặp này trước khi phân tích tiếp.

In [ ]:
id_counts = df_b2['ID'].value_counts()
print(id_counts.head())

ID
89f8f447-fca8    2
7c9b7473-cc2f    2
90330d02-82d9    2
75938fec-e5ec    2
2a793ecf-05c6    2
Name: count, dtype: int64


Dựa trên kết quả `value_counts()`, chúng ta thấy có một số ID xuất hiện 2 lần (ví dụ như các ID hàng đầu trong danh sách). Điều này xác nhận rằng tập dữ liệu đang có các mẫu bị trùng lặp thông tin định danh, ảnh hưởng đến tính toàn vẹn của dữ liệu. Chúng ta cần kiểm tra xem các dòng trùng ID này có trùng hoàn toàn dữ liệu tài chính hay không để có hướng xử lý phù hợp.

## **2.5. pandas.DataFrame.loc**

Phương thức `.loc` là một thuộc tính của DataFrame dùng để truy cập một nhóm các hàng và cột dựa trên nhãn (label) hoặc một mảng boolean.

Khác với các phương thức dựa trên vị trí, `.loc chủ yếu dựa vào tên gọi của chỉ mục (index) để xác định dữ liệu cần truy xuất.

Bạn có thể truyền vào một nhãn đơn lẻ, ví dụ như số `5` hoặc chuỗi `'a'`; lưu ý rằng nếu truyền số `5`, Pandas sẽ tìm hàng có nhãn là `5` chứ không phải hàng ở vị trí thứ năm.

Một danh sách hoặc mảng các nhãn như `['a', 'b', 'c']` cũng được chấp nhận để lấy ra nhiều hàng hoặc cột cùng lúc.

Khi thực hiện cắt lát (slicing) bằng nhãn, ví dụ `'a':'f'`, bạn cần đặc biệt lưu ý rằng cả điểm bắt đầu và điểm kết thúc đều sẽ được bao gồm trong kết quả trả về.

Phương thức này cũng hỗ trợ mảng boolean có cùng độ dài với trục đang được cắt, giúp bạn lọc dữ liệu theo các điều kiện logic một cách linh hoạt.

Bên cạnh các nhãn cố định, bạn có thể truyền vào một hàm (callable) có một đối số để trả về kết quả lập chỉ mục hợp lệ.

Ngoài việc truy xuất, `.loc` còn được sử dụng mạnh mẽ để thiết lập giá trị cho các mục, toàn bộ hàng hoặc toàn bộ cột khớp với điều kiện chỉ định.

Một đặc điểm quan trọng khi gán dữ liệu bằng `.loc` là Pandas sẽ tự động căn chỉnh (align) các nhãn chỉ mục thay vì gán theo thứ tự vị trí của dữ liệu.

Nếu bạn cố gắng truy cập một nhãn không tồn tại trong DataFrame, hệ thống sẽ trả về lỗi `KeyError`.

Đối với các cấu trúc dữ liệu phức tạp, `.loc` cũng hỗ trợ làm việc với MultiIndex (chỉ mục đa cấp) thông qua việc truyền vào các bộ giá trị (tuples).

## **2.6. Bài tập 2: Tiếp tục Xác minh tính toàn vẹn của dữ liệu**

### Đánh dấu và lập danh sách các ID bị trùng

In [ ]:
# Tạo mặt nạ boolean để đánh dấu các ID xuất hiện đúng 2 lần
dupe_mask = (id_counts == 2)

# Hiển thị 5 phần tử đầu tiên của mặt nạ
print("5 phần tử đầu tiên của dupe_mask:")
print(dupe_mask[0:5])

# Trích xuất các ID từ chỉ mục (index) nơi dupe_mask là True và chuyển thành danh sách
dupe_ids = list(id_counts.index[dupe_mask])

5 phần tử đầu tiên của dupe_mask:
ID
89f8f447-fca8    True
7c9b7473-cc2f    True
90330d02-82d9    True
75938fec-e5ec    True
2a793ecf-05c6    True
Name: count, dtype: bool


### Kiểm tra số lượng phần tử của `dupe_ids`

In [ ]:
# Kiểm tra số lượng ID bị trùng
len(dupe_ids)

313

Số lượng phần tử trong danh sách `dupe_ids` là 313.

Kết quả này hoàn toàn khớp với logic tính toán trước đó: lấy tổng số 30.000 dòng trừ đi 29.687 giá trị ID duy nhất sẽ dư ra 313 bản ghi bị lặp lại.

Điều này xác nhận rằng có đúng 313 mã định danh khách hàng đang xuất hiện nhiều hơn một lần trong tập dữ liệu ban đầu.

### Kiểm tra dữ liệu thực tế của các ID bị trùng

In [ ]:
# Lọc toàn bộ các dòng có ID nằm trong 3 ID bị trùng đầu tiên để quan sát
df_b2.loc[df_b2['ID'].isin(dupe_ids[0:3]), :].head(10)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
5033,89f8f447-fca8,320000,2,2,1,32,0,0,0,0,...,169371,172868,150827,8000,8000,5500,6100,6000,5000,0
5133,89f8f447-fca8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15879,7c9b7473-cc2f,90000,2,1,1,29,0,0,0,0,...,27751,20292,14937,2967,2007,1429,1092,412,263,0
15979,7c9b7473-cc2f,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29646,90330d02-82d9,70000,1,2,1,29,0,0,0,0,...,10694,27908,11192,2009,1404,3016,20001,2000,5002,0
29746,90330d02-82d9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Ý nghĩa của đoạn code:

Đoạn mã trên sử dụng phương thức `.isin()` để kiểm tra xem giá trị ID của mỗi dòng trong DataFrame có nằm trong danh sách 3 mã ID bị trùng đầu tiên (`dupe_ids[0:3]`) hay không.

Phương thức .loc[..., :] được sử dụng để trích xuất tất cả các hàng thỏa mãn điều kiện và hiển thị đầy đủ tất cả các cột của các hàng đó.

Việc sử dụng `.head(10)` giúp chúng ta quan sát trực quan các bản ghi này để so sánh xem các thông tin tài chính và nhân khẩu học của cùng một ID có giống hệt nhau hay có sự khác biệt.

### Nhận xét cách xóa các dòng trùng lặp

Nhận xét về cách xử lý và xóa các dòng trùng lặp:

* Qua kiểm tra sơ bộ, chúng ta thường thấy các cặp ID bị trùng có một dòng chứa dữ liệu hợp lệ và một dòng chứa toàn bộ giá trị bằng 0 (dữ liệu rác).

* Một cách tiếp cận hiệu quả là tạo ra một mặt nạ boolean (boolean mask) để lọc ra những dòng có tất cả các cột dữ liệu đều bằng 0 và loại bỏ chúng.

* Ngoài ra, chúng ta có thể sử dụng hàm `df.drop_duplicates()` của thư viện Pandas để xóa bỏ các bản ghi trùng lặp một cách nhanh chóng.

* Sau khi xóa, chúng ta cần thực hiện kiểm tra lại một lần nữa để đảm bảo số lượng dòng còn lại trong DataFrame phải bằng đúng số lượng ID duy nhất (29.687 dòng).

* Việc làm sạch này là bước bắt buộc để đảm bảo mô hình dự đoán không bị sai lệch do dữ liệu nhiễu hoặc dữ liệu ảo gây ra.

## **2.6. pandas.DataFrame.iloc**

* Phương thức `.iloc` là một thuộc tính của DataFrame dùng để lựa chọn dữ liệu dựa hoàn toàn vào vị trí số nguyên (integer-position) của các hàng và cột.
* Khác với .loc dựa trên tên nhãn, `.iloc` tuân theo vị trí vật lý trong bảng dữ liệu, bắt đầu từ chỉ số 0 cho đến length-1 của trục.
* Bạn có thể sử dụng một số nguyên đơn lẻ để truy cập một hàng hoặc một giá trị cụ thể, ví dụ như df.`iloc[5]`.
* Phương thức này cũng chấp nhận một danh sách hoặc một mảng các số nguyên để lấy ra nhiều hàng/cột cùng lúc, ví dụ như `df.iloc[[4, 3, 0]]`.
* Khi sử dụng đối tượng cắt lát (slice) như `1:7`, `.iloc` tuân thủ quy tắc chuẩn của Python và NumPy, nghĩa là điểm bắt đầu được bao gồm nhưng điểm kết thúc sẽ bị loại bỏ.
* Ngoài chỉ số số nguyên, bạn có thể truyền vào một mảng boolean để lọc dữ liệu theo các điều kiện logic nhất định.Một hàm có thể gọi (callable) cũng được chấp nhận làm đầu vào, điều này rất hữu ích khi bạn muốn thực hiện các thao tác chuỗi phương thức (method chaining) mà không có tham chiếu trực tiếp đến đối tượng gọi hàm.
* `.iloc` sẽ báo lỗi `IndexError` nếu chỉ mục yêu cầu nằm ngoài phạm vi của bảng dữ liệu, ngoại trừ các trường hợp cắt lát cho phép lập chỉ mục ngoài phạm vi.
* Bạn có thể lập chỉ mục cho cả hai trục (hàng và cột) bằng cách truyền vào một bộ gồm chỉ số hàng và cột, ví dụ `df.iloc[0, 1]`.
* So với các phương thức khác, `.iloc` cung cấp cách truy cập dữ liệu nhanh chóng và chính xác khi bạn đã biết rõ vị trí thứ tự của các hàng hoặc cột trong DataFrame.

## **2.7. Bài tập 3: Loại bỏ các dòng trùng lặp của dữ liệu**

In [ ]:
df_numeric = df_b2.copy()
for col in df_numeric.columns[1:]:
    df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce').fillna(0)

In [ ]:
df_zero_mask = df_numeric == 0

feature_zero_mask = df_zero_mask.iloc[:, 1:].all(axis=1)

df_clean_1 = df_b2.loc[~feature_zero_mask, :].copy()

print(f"Số dòng ban đầu: {df_b2.shape[0]}")
print(f"Số dòng sau khi làm sạch: {df_clean_1.shape[0]}")
print(f"Số lượng ID duy nhất: {df_clean_1['ID'].nunique()}")

Số dòng ban đầu: 30000
Số dòng sau khi làm sạch: 29685
Số lượng ID duy nhất: 29685


Dòng mã `df_zero_mask = df == 0` tạo ra một bảng mặt nạ có cùng kích thước với dữ liệu gốc, trong đó mỗi ô sẽ trả về giá trị `True` nếu dữ liệu tại đó bằng 0 và `False` nếu ngược lại.

Lệnh `df_zero_mask.iloc[:, 1:]` giúp chúng ta loại bỏ cột đầu tiên (cột ID) ra khỏi quá trình kiểm tra, vì các dòng trùng lặp vẫn có ID hợp lệ nên chúng ta chỉ cần kiểm tra các biến đặc trưng khác.

Phương thức `.all(axis=1)` thực hiện kiểm tra theo từng hàng; nó sẽ trả về `True` chỉ khi toàn bộ các cột đặc trưng trong hàng đó đều là số 0.

Ký hiệu ngã (`~`) trong lệnh `df.loc[~feature_zero_mask, :]` đóng vai trò là toán tử phủ định (NOT), giúp chúng ta chọn lọc và giữ lại những hàng có ít nhất một giá trị khác 0.

Cuối cùng, lệnh `.copy()` được sử dụng để tạo ra một bản sao độc lập cho `df_clean_1`, tránh việc thay đổi dữ liệu trên DataFrame mới làm ảnh hưởng đến dữ liệu gốc trong bộ nhớ.

Sau khi thực thi, tập dữ liệu mới `df_clean_1` còn lại 29.687 dòng và 25 cột.

Chúng ta đã loại bỏ thành công 313 dòng dữ liệu trống. Con số này hoàn toàn khớp với số lượng ID bị trùng đã xác định ở Bài tập 2.

Hiện tại, số lượng dòng trong DataFrame đã bằng đúng với số lượng ID duy nhất, xác nhận rằng tính toàn vẹn của dữ liệu về mặt định danh đã được khôi phục.

In [ ]:
df_clean_1.to_csv('credit_card_clean_1.csv', index=False)